**Importing Necessary Liabraries :-**

In [123]:
# Import Liabraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine
import urllib.parse
import psycopg2
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

**Connecting Python to PostgreSQL :-**

In [124]:
username ='postgres'
password = urllib.parse.quote_plus('Canada@040201')
host = 'localhost'
port = '5432'
database = 'Taxi_Analysis_db'
engine = create_engine(f'postgresql+psycopg2://{username}:{password}@{host}:{port}/{database}')

**Loading CSV Data to PostgreSQL :-**

In [125]:
# file_path = r"C:\Users\hp\OneDrive\Documents\DA Course\End to End Projects\Project 6\yellow_tripdata.csv"

# table_name = "yellow_tripdata"

# df = pd.read_csv(file_path, parse_dates=["tpep_pickup_datetime", "tpep_dropoff_datetime"],low_memory=False)
# df.to_sql(table_name, engine, if_exists="replace", index=False)

# print(f"Loaded {table_name}: {len(df)} rows")

**Importing Data in python from PostgreSQL :-**

In [126]:
conn = psycopg2.connect(
    dbname="Taxi_Analysis_db",
    user="postgres",
    password="Canada@040201",
    host="localhost",
    port="5432"
)

sql_query="""
          SELECT table_name AS name FROM information_schema.tables
          WHERE table_schema = 'public'
"""

In [127]:
table_name = "yellow_tripdata"
df = pd.read_sql(f"SELECT * FROM {table_name}", conn) 

In [128]:
df.shape

(4090836, 20)

**Print tables and columns name :-**

In [129]:
sql_query = """
SELECT table_name AS name FROM information_schema.tables
WHERE table_schema = 'public'
"""
tables = pd.read_sql(sql_query, conn)

# For each table, get and print its column names
for table_name in tables['name']:
    print(f"\nTable Name: {table_name}")

    columns_query = f"""
    SELECT column_name AS name
    FROM information_schema.columns
    WHERE table_name = '{table_name}'
    """
    columns = pd.read_sql(columns_query, conn)
    print('Columns:', columns['name'].tolist())

conn.close()


Table Name: yellow_tripdata
Columns: ['cbd_congestion_fee', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'RatecodeID', 'VendorID', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'Airport_fee', 'store_and_fwd_flag']

Table Name: taxi_zone_lookup
Columns: ['LocationID', 'Borough', 'Zone', 'service_zone']

Table Name: dim_zone
Columns: ['LocationID', 'Borough', 'Zone', 'service_zone']

Table Name: dim_vendor
Columns: ['VendorID', 'Vendor_Names']

Table Name: dim_rate_code
Columns: ['RatecodeID', 'Rate_Code']

Table Name: dim_payment_type
Columns: ['payment_type', 'Payment_Method']

Table Name: dim_store_and_fwd_flag
Columns: ['store_and_fwd_flag', 'Flags']

Table Name: fact_yellow_trip
Columns: ['is_distance_outlier', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'RatecodeID', 'Ven

**Data Overview in Table 1 (Yellow_Tripdata Table) :-**

**Data Dictionary :-**

1)-**VendorID**              : Code that tells us which taxi company or system recorded the trip.

2)-**tpep_pickup_datetime**     : The exact date and time when the passenger was picked up.

3)-**tpep_dropoff_datetime**   : The exact date and time when the passenger was dropped off.

4)-**passenger_count**         : Number of passengers in the trip. Entered by the driver, so may not always be accurate.

5)-**trip_distance**            : Distance of the trip in miles, measured by the taxi meter.

6)-**RatecodeID**           : Tells us which fare/rate type was used.
1 = Standard, 2 = JFK, 3 = Newark, 4 = Nassau/Westchester, 5 = Negotiated fare, 6 = Group ride, 99 = Unknown.
JFK refers to John F. Kennedy International Airport in New York City.
                          
7)-**store_and_fwd_flag**       : Whether trip data was stored in the taxi first and sent to the server later due to a network problem. Y = Yes, N = No.

8)-**PULocationID**             : ID of the pickup location/taxi zone (matches NYC taxi zone lookup table).

9)-*DOLocationID*              : ID of the drop-off location/taxi zone.

10)-**payment_type**            : How the passenger paid.
1 = Credit card, 2 = Cash, 3 = No charge, 4 = Dispute.Other codes represent less common cases.

11)-**fare_amount**               : The basic taxi fare charged, before extra charges, taxes, tips, and other fees.

12)-**extra**                    : Extra charges added to the fare, such as rush-hour or overnight charges.

13)-**mta_tax**                 : A tax collected for the Metropolitan Transportation Authority (MTA).

14)-**tip_amount**                 : Recorded tip amount. Card tips are recorded by the system; cash tips are generally not recorded in this field.

15)-**tolls_amount**              : Total amount of tolls paid during the trip.

16)-**improvement_surcharge**     : A small surcharge used to support taxi-related improvements.

17)-**total_amount**               : Total recorded amount charged for the trip, including fare, extras,
taxes, tips, tolls, and applicable surcharges. It is not profit.

18)-**congestion_surcharge**        : A congestion-related extra charge recorded for eligible trips
under the applicable NYC taxi surcharge rules.

19)-**Airport_fee**                 : An extra airport-related fee charged when the trip meets
the applicable airport fee rules.

20)-**cbd_congestion_fee**            : A congestion fee related to NYC's Central Business District (CBD)
congestion pricing rules.

In [130]:
# Checking Top 5 rows of the dataset
df.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,2,2026-05-01 00:04:59,2026-05-01 00:32:48,1.00,7.51,1.00,N,138,37,2,35.90,6.00,0.50,0.00,0.00,1.00,45.40,0.00,2.00,0.00
1,2,2026-05-01 00:37:05,2026-05-01 00:59:47,1.00,6.14,1.00,N,138,237,1,27.50,6.00,0.50,9.56,0.00,1.00,49.81,2.50,2.00,0.75
2,1,2026-05-01 00:34:05,2026-05-01 00:54:02,1.00,2.40,1.00,N,249,232,1,19.10,4.25,0.50,3.73,0.00,1.00,28.58,2.50,0.00,0.75
3,1,2026-05-01 00:55:07,2026-05-01 01:02:43,0.00,1.20,1.00,N,232,114,1,9.30,4.25,0.50,3.00,0.00,1.00,18.05,2.50,0.00,0.75
4,7,2026-05-01 00:44:13,2026-05-01 00:44:13,2.00,0.86,1.00,N,140,237,1,7.20,0.00,0.50,2.44,0.00,1.00,14.64,2.50,0.00,0.00


In [131]:
# Checking Bottom 5 rows of the dataset
df.tail()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
4090831,2,2026-05-31 23:35:12,2026-05-31 23:51:04,NaN,5.11,NaN,NaN,74,243,0,17.19,0.00,0.50,0.00,0.00,1.00,18.69,NaN,NaN,0.00
4090832,1,2026-05-31 23:48:01,2026-06-01 00:05:57,NaN,8.80,NaN,NaN,129,130,0,27.98,0.00,0.50,0.00,0.00,1.00,33.48,NaN,NaN,0.00
4090833,2,2026-05-31 23:48:50,2026-06-01 00:03:52,NaN,6.47,NaN,NaN,48,116,0,24.12,0.00,0.50,0.00,0.00,1.00,28.87,NaN,NaN,0.75
4090834,2,2026-05-31 23:23:13,2026-05-31 23:38:35,NaN,2.94,NaN,NaN,113,256,0,18.20,0.00,0.50,0.00,0.00,1.00,22.95,NaN,NaN,0.75
4090835,2,2026-05-31 23:41:51,2026-05-31 23:47:32,NaN,0.83,NaN,NaN,255,112,0,7.02,0.00,0.50,0.00,0.00,1.00,8.52,NaN,NaN,0.00


In [132]:
# Checking 10 Random 10 Rows  of the dataset
df.sample(10)

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
3273055,2,2026-05-06 10:07:25,2026-05-06 10:45:04,NaN,4.73,NaN,NaN,224,263,0,34.84,0.00,0.50,0.00,0.00,1.00,38.84,NaN,NaN,0.00
2698226,2,2026-05-27 17:27:29,2026-05-27 18:09:23,1.00,9.99,1.00,N,138,65,2,46.40,7.50,0.50,0.00,0.00,1.00,57.40,0.00,2.00,0.00
2723798,2,2026-05-27 21:12:43,2026-05-27 21:17:32,1.00,1.45,1.00,N,237,236,1,7.90,1.00,0.50,2.58,0.00,1.00,15.48,2.50,0.00,0.00
3865391,1,2026-05-24 18:18:52,2026-05-24 18:38:10,NaN,3.00,NaN,NaN,79,66,0,17.75,0.00,0.50,0.00,0.00,1.00,23.94,NaN,NaN,0.75
1348103,2,2026-05-13 22:32:27,2026-05-13 22:46:01,1.00,2.19,1.00,N,237,107,1,14.20,1.00,0.50,3.99,0.00,1.00,23.94,2.50,0.00,0.75
3709906,2,2026-05-19 20:50:12,2026-05-19 21:06:21,NaN,2.44,NaN,NaN,186,79,0,22.20,0.00,0.50,0.00,0.00,1.00,26.95,NaN,NaN,0.75
2176445,2,2026-05-21 14:57:36,2026-05-21 15:27:03,1.00,4.76,1.00,N,163,249,2,28.90,0.00,0.50,0.00,0.00,1.00,33.65,2.50,0.00,0.75
1890994,2,2026-05-18 22:28:14,2026-05-18 22:55:30,1.00,5.17,1.00,N,234,49,1,28.90,1.00,0.50,6.93,0.00,1.00,41.58,2.50,0.00,0.75
933913,2,2026-05-09 22:23:47,2026-05-09 22:48:30,1.00,1.99,1.00,N,144,170,1,20.50,1.00,0.50,2.00,0.00,1.00,28.25,2.50,0.00,0.75
2106452,2,2026-05-20 20:25:52,2026-05-20 20:38:37,1.00,1.34,1.00,N,142,140,1,12.80,1.00,0.50,3.56,0.00,1.00,21.36,2.50,0.00,0.00


In [133]:
# Checking the shape of the dataset (Number of rows and columns)
print('Total Rows in Customer Table :',df.shape[0])
print('Total Columns in Customer Table :',df.shape[1])

Total Rows in Customer Table : 4090836
Total Columns in Customer Table : 20


In [134]:
# Getting information about our dataset like total rows,
# columns,datatypes of each column and memory requirement.
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4090836 entries, 0 to 4090835
Data columns (total 20 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   VendorID               int64         
 1   tpep_pickup_datetime   datetime64[us]
 2   tpep_dropoff_datetime  datetime64[us]
 3   passenger_count        float64       
 4   trip_distance          float64       
 5   RatecodeID             float64       
 6   store_and_fwd_flag     str           
 7   PULocationID           int64         
 8   DOLocationID           int64         
 9   payment_type           int64         
 10  fare_amount            float64       
 11  extra                  float64       
 12  mta_tax                float64       
 13  tip_amount             float64       
 14  tolls_amount           float64       
 15  improvement_surcharge  float64       
 16  total_amount           float64       
 17  congestion_surcharge   float64       
 18  Airport_fee            float64   

In [135]:
# Checking % for missing dataset in each column
df.isnull().mean()*100

VendorID                 0.00
tpep_pickup_datetime     0.00
tpep_dropoff_datetime    0.00
passenger_count         23.35
trip_distance            0.00
RatecodeID              23.35
store_and_fwd_flag      23.35
PULocationID             0.00
DOLocationID             0.00
payment_type             0.00
fare_amount              0.00
extra                    0.00
mta_tax                  0.00
tip_amount               0.00
tolls_amount             0.00
improvement_surcharge    0.00
total_amount             0.00
congestion_surcharge    23.35
Airport_fee             23.35
cbd_congestion_fee       0.00
dtype: float64

In [136]:
# Checking not null values in each column
df.notnull().sum()

VendorID                 4090836
tpep_pickup_datetime     4090836
tpep_dropoff_datetime    4090836
passenger_count          3135465
trip_distance            4090836
RatecodeID               3135465
store_and_fwd_flag       3135465
PULocationID             4090836
DOLocationID             4090836
payment_type             4090836
fare_amount              4090836
extra                    4090836
mta_tax                  4090836
tip_amount               4090836
tolls_amount             4090836
improvement_surcharge    4090836
total_amount             4090836
congestion_surcharge     3135465
Airport_fee              3135465
cbd_congestion_fee       4090836
dtype: int64

In [137]:
# Checking total duplicated values available in the dataset.
duplicate_values=df.duplicated().sum()
print('Total Duplicate Values :',duplicate_values)

Total Duplicate Values : 0


In [138]:
# Create a copy of dataset before analysing.
df1=df.copy()

**Data Cleaning in Table 1 (Yellow_Tripdata Table) :-**

In [139]:
# What to fix in table 1 (Yellow Trip Data).
# 1)-Need to change data type from float to int for columsn-passenger_count/RatecodeID/improvement_surcharge/Airport_fee.
# 2)-Need to check negative values in improvement_surcharge/fare_amount/extra/mta_tax/total_amount/congestion_surcharge/cbd_congestion_fee columns.
# 3)-Need to check value(0) in trip_distance/passenger_count column.
# 4)-Need to check extreme values in fare_amount and trip_distance.
# 5)- Need to remove values(264,265) from DOLocationID column.
# 6)-Need to handle null values in columns-passenger_count/RatecodeID/store_and_fwd_flag/congestion_surcharge/Airport_fee.

In [140]:
# 1)-Need to change data type from float to int for columsn-passenger_count/RatecodeID/improvement_surcharge/Airport_fee.
cols=['passenger_count','RatecodeID']
df[cols]=df[cols].astype('Int64')

In [141]:
df[cols].dtypes

passenger_count    Int64
RatecodeID         Int64
dtype: object

In [142]:
# 2)-Need to check negative values in improvement_surcharge/fare_amount/extra/mta_tax/total_amount/tip_amount/Airport_fee/congestion_surcharge/cbd_congestion_fee columns.
money_cols = ['fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount',
              'improvement_surcharge', 'total_amount', 'congestion_surcharge',
              'Airport_fee', 'cbd_congestion_fee']

In [143]:
# Flag rows that are negative AND have zero trip distance = likely voided (no real ride happened)
df['is_cancelled'] = (df[money_cols] < 0).any(axis=1) & (df['trip_distance'] == 0)

In [144]:
# Check how many got flagged
print(df['is_cancelled'].value_counts())

is_cancelled
False    4089064
True        1772
Name: count, dtype: int64


In [145]:
# Flag rows that are disputed AND have a negative amount
df['is_dispute_adjustment'] = (df['payment_type'] == 4) & (df[money_cols] < 0).any(axis=1)

In [146]:
# Check how many got flagged
print(df['is_dispute_adjustment'].value_counts())

is_dispute_adjustment
False    4081792
True        9044
Name: count, dtype: int64


First, we will create a variable to store all negative values (Dispute Adjustment + Cancelled Ride) present in money_cols. If needed this lets us track both voided_refund and disputed_adjustment separately without losing any rows from the dataset.

Cancelled Ride = Negative Fare Amount + 0 miles Distance → **ride never happened, truly voided.**

Disputed Adjustment = payment_type = 4 + negative amount → **real ride happened, charge was disputed/reversed afterward**

If our goal is to "analyze how taxi rides perform" — trip counts, real revenue, Average fare — then these voided/disputed rows aren't real completed rides, so including them just adds noise and skews your numbers.

**Note**- We can't take absolute value as it will turns a cancellation/dispute into a fake positive fare, distorting revenue numbers/trip count.

In [147]:
# Store negative-value rows before removing them for future reference and review.

numeric_cols = df.select_dtypes(include='number').columns

negative_cols = numeric_cols[(df[numeric_cols]<0).any()]

negative_values = df.loc[(df[negative_cols]<0).any(axis=1), negative_cols]
negative_values

,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
26,-12.80,-1.00,-0.50,0.00,0.00,-1.00,-18.55,-2.50,0.00,-0.75
107,-3.70,-1.00,-0.50,0.00,0.00,-1.00,-9.45,-2.50,0.00,-0.75
274,-30.30,-1.00,-0.50,0.00,0.00,-1.00,-36.05,-2.50,0.00,-0.75
386,-12.10,-1.00,-0.50,0.00,0.00,-1.00,-17.85,-2.50,0.00,-0.75
456,-24.00,-1.00,-0.50,0.00,0.00,-1.00,-29.75,-2.50,0.00,-0.75
...,...,...,...,...,...,...,...,...,...,...
3991915,84.15,-0.75,0.00,0.00,14.79,1.00,100.69,NaN,NaN,0.75
4023323,-2.59,0.00,0.50,3.32,0.00,1.00,5.48,NaN,NaN,0.75
4038508,-1.12,0.00,0.50,1.18,0.00,1.00,4.81,NaN,NaN,0.75
4048345,-0.95,0.00,0.50,0.00,29.58,1.00,32.63,NaN,NaN,0.00


In [148]:
# Drop Negative value columns
df_clean = df.drop(
    df.index[(df[negative_cols] < 0).any(axis=1)],
).copy()

In [149]:
(df_clean[negative_cols] < 0).any(axis=1).sum()

np.int64(0)

In [150]:
print("Original rows:", df.shape[0])
print("Clean rows (after excluding disputes + cancelled):", df_clean.shape[0])

Original rows: 4090836
Clean rows (after excluding disputes + cancelled): 4075883


In [151]:
pct_negative = ((df.shape[0] - df_clean.shape[0]) / df.shape[0]) * 100
print('Percentage of Negative Records :',round(pct_negative,2),'%')

Percentage of Negative Records : 0.37 %


In [152]:
# 3)-Need to check value(0) in trip_distance/passenger_count column.
df_clean.columns

Index(['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime',
       'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag',
       'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra',
       'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge',
       'total_amount', 'congestion_surcharge', 'Airport_fee',
       'cbd_congestion_fee', 'is_cancelled', 'is_dispute_adjustment'],
      dtype='str')

**trip_distance = 0** are real trips where the distance meter/GPS just didn't record properly — but a fare was still charged.Dropping all of these rows would throw away real revenue data. 

We should replace 0 with NaN (missing).so it doesn't quietly get counted as "0 miles" in distance-based averages later, while the fare amount stays intact for revenue analysis.

In [153]:
# Replace trip_distance = 0 with missing (NA)
df_clean.loc[df_clean['trip_distance'] == 0, 'trip_distance'] = pd.NA

**passenger_count = 0**  A taxi trip can't have 0 passengers — this is a sensor/entry error, not a valid case, there's no business reason for 0 passengers.

We should replace 0 with missing, rather than guessing/imputing a passenger count — since we can't know the true count.
Imputing with the mode (1) would fabricate data we don't actually know — better to leave it honestly missing.



In [154]:
# Replace passenger_count = 0 with missing (NA)
df_clean.loc[df_clean['passenger_count'] == 0, 'passenger_count'] = pd.NA

In [155]:
# 4)-Need to check extreme values in fare_amount.
df_clean.columns

Index(['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime',
       'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag',
       'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra',
       'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge',
       'total_amount', 'congestion_surcharge', 'Airport_fee',
       'cbd_congestion_fee', 'is_cancelled', 'is_dispute_adjustment'],
      dtype='str')

In [156]:
df_clean['trip_distance'].isnull().sum()

np.int64(111259)

In [157]:
df_clean['fare_amount'].isnull().sum()

np.int64(0)

In [158]:
# Calculate fare per mile (avoid divide-by-zero)
df_clean['fare_per_mile'] = df_clean['fare_amount'] / df_clean['trip_distance'].replace(0, pd.NA)

**Note** -> replace(0, pd.NA) does: it first swaps every 0 in trip_distance with NA (missing), so the division becomes fare_amount / NA, which pandas handles cleanly by returning NA instead of inf.

In [159]:
threshold = df_clean['fare_per_mile'].quantile(0.99)
print('Threshold :',threshold)

Threshold : 46.25


In [160]:
zero_dist_threshold = df_clean[(df_clean['trip_distance'].isna()) & (df_clean['fare_amount']>0)]['fare_amount'].quantile(0.99)
zero_dist_threshold

np.float64(120.0)

In [161]:
# Exclude JFK (2) and Negotiated (5) fares from Fare outliers.
df_clean['is_fare_outlier'] = (
    (df_clean['fare_per_mile'] > threshold) |
    ((df_clean['trip_distance'].isna()) & (df_clean['fare_amount'] > zero_dist_threshold))
) & (~df_clean['RatecodeID'].isin([2, 5]))   

print("Total flagged fare outliers :", df_clean['is_fare_outlier'].sum())

Total flagged fare outliers : 31982


In [162]:
# Checking all fare outliers.
pd.set_option('display.max_columns', None)
df_clean[df_clean['is_fare_outlier']]

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,is_cancelled,is_dispute_adjustment,fare_per_mile,is_fare_outlier
416,2,2026-05-01 00:33:58,2026-05-01 00:53:43,1,0.15,1,N,100,100,1,16.30,1.00,0.50,4.41,0.00,1.00,26.46,2.50,0.00,0.75,False,False,108.67,True
1249,2,2026-05-01 00:11:04,2026-05-01 00:14:34,1,0.04,1,N,138,265,1,5.10,6.00,0.50,18.00,0.00,1.00,32.60,0.00,2.00,0.00,False,False,127.50,True
1390,2,2026-05-01 00:51:19,2026-05-01 00:54:26,1,0.07,1,N,158,249,4,5.10,1.00,0.50,0.00,0.00,1.00,10.85,2.50,0.00,0.75,False,False,72.86,True
1811,2,2026-05-01 00:56:20,2026-05-01 00:58:25,1,0.09,1,N,164,164,1,4.40,1.00,0.50,2.03,0.00,1.00,12.18,2.50,0.00,0.75,False,False,48.89,True
1922,2,2026-05-01 00:19:55,2026-05-01 00:20:54,1,0.01,1,N,79,79,2,3.00,1.00,0.50,0.00,0.00,1.00,8.75,2.50,0.00,0.75,False,False,300.00,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4089716,2,2026-05-31 22:35:54,2026-05-31 22:37:16,<NA>,0.13,<NA>,NaN,181,181,0,8.72,0.00,0.50,0.00,0.00,1.00,15.22,NaN,NaN,0.00,False,False,67.08,True
4090350,2,2026-05-31 23:27:44,2026-05-31 23:30:58,<NA>,0.38,<NA>,NaN,246,143,0,26.07,0.00,0.50,0.00,0.00,1.00,30.82,NaN,NaN,0.75,False,False,68.61,True
4090355,2,2026-05-31 23:06:53,2026-05-31 23:07:56,<NA>,0.04,<NA>,NaN,68,186,0,15.65,0.00,0.50,0.00,0.00,1.00,20.40,NaN,NaN,0.75,False,False,391.25,True
4090817,2,2026-05-31 23:01:04,2026-05-31 23:04:28,<NA>,0.41,<NA>,NaN,152,116,0,19.12,0.00,0.50,0.00,0.00,1.00,20.62,NaN,NaN,0.00,False,False,46.63,True


In [163]:
# Check RatecodeID breakdown for all flagged fare outliers
df_clean['fare_amount'].isnull().sum()

np.int64(0)

In [164]:
# Replace fare_amount with NaN for flagged outliers (keep the rows)
df_clean.loc[df_clean['is_fare_outlier'], 'fare_amount'] = np.nan

In [165]:
# Check RatecodeID breakdown for all flagged fare outliers
df_clean['fare_amount'].isnull().sum()

np.int64(31982)

In [166]:
df_clean.columns

Index(['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime',
       'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag',
       'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra',
       'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge',
       'total_amount', 'congestion_surcharge', 'Airport_fee',
       'cbd_congestion_fee', 'is_cancelled', 'is_dispute_adjustment',
       'fare_per_mile', 'is_fare_outlier'],
      dtype='str')

In [167]:
df_clean[df_clean['tip_amount']<0]

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,is_cancelled,is_dispute_adjustment,fare_per_mile,is_fare_outlier


In [168]:
print("df rows:", df.shape[0])
print("df_clean rows:", df_clean.shape[0])
print("Outliers removed:", df.shape[0] - df_clean.shape[0])

df rows: 4090836
df_clean rows: 4075883
Outliers removed: 14953


In [169]:
# 5)- Need to remove values(264,265) from DOLocationID column.
df_clean.columns

Index(['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime',
       'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag',
       'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra',
       'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge',
       'total_amount', 'congestion_surcharge', 'Airport_fee',
       'cbd_congestion_fee', 'is_cancelled', 'is_dispute_adjustment',
       'fare_per_mile', 'is_fare_outlier'],
      dtype='str')

In [170]:
# Flag rows with invalid location IDs (valid NYC zones are 1-263)
df_clean['is_invalid_location'] = (
    (df_clean['PULocationID'] < 1) | (df_clean['PULocationID'] > 263) |
    (df_clean['DOLocationID'] < 1) | (df_clean['DOLocationID'] > 263)
)

print("Invalid location rows:", df_clean['is_invalid_location'].sum())

Invalid location rows: 25020


In [171]:
df_clean = df_clean[~df_clean['is_invalid_location']].copy()
print("Rows after removing invalid locations:", df_clean.shape[0])

Rows after removing invalid locations: 4050863


In [172]:
# 6)-Need to handle null values in columns-passenger_count/RatecodeID/store_and_fwd_flag/congestion_surcharge/Airport_fee.
df_clean.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,is_cancelled,is_dispute_adjustment,fare_per_mile,is_fare_outlier,is_invalid_location
0,2,2026-05-01 00:04:59,2026-05-01 00:32:48,1,7.51,1,N,138,37,2,35.90,6.00,0.50,0.00,0.00,1.00,45.40,0.00,2.00,0.00,False,False,4.78,False,False
1,2,2026-05-01 00:37:05,2026-05-01 00:59:47,1,6.14,1,N,138,237,1,27.50,6.00,0.50,9.56,0.00,1.00,49.81,2.50,2.00,0.75,False,False,4.48,False,False
2,1,2026-05-01 00:34:05,2026-05-01 00:54:02,1,2.40,1,N,249,232,1,19.10,4.25,0.50,3.73,0.00,1.00,28.58,2.50,0.00,0.75,False,False,7.96,False,False
3,1,2026-05-01 00:55:07,2026-05-01 01:02:43,<NA>,1.20,1,N,232,114,1,9.30,4.25,0.50,3.00,0.00,1.00,18.05,2.50,0.00,0.75,False,False,7.75,False,False
4,7,2026-05-01 00:44:13,2026-05-01 00:44:13,2,0.86,1,N,140,237,1,7.20,0.00,0.50,2.44,0.00,1.00,14.64,2.50,0.00,0.00,False,False,8.37,False,False


**Passenger_Count** --> Leave as missing (NA) as there is no way to know the real count.

In [173]:
df_clean['RatecodeID'].value_counts()

RatecodeID
1     2820347
99     140766
2       93686
5       31461
3       11527
4        1030
6           1
Name: count, dtype: Int64

In [174]:
df_clean['RatecodeID'].isnull().sum()

np.int64(952045)

**RatecodeID** --> Category with an official "unknown" code. So fill with 99 as TLC's own data dictionary defines 99 as "Null/unknown".

In [175]:
df_clean['RatecodeID'] = df_clean['RatecodeID'].fillna(99)

**Store_and_fwd_flag** --> Category (Y/N). Can't assume Y or N. So filing with "Unknown".

In [176]:
df_clean['store_and_fwd_flag'] = df_clean['store_and_fwd_flag'].fillna('Unknown')

**Congestion_surcharge** --> Without knowing the trip's actual route, we can't tell if it crossed into the surcharged zone or not. Filling with 0 would silently claim "no surcharge applied," which could easily be wrong for a real trip. So we must leave null values as null.

**Airport_fee** --> Airport_fee only applies to pickups at JFK (LocationID 132) or LaGuardia (LocationID 138), 
per TLC's official data dictionary. This means a missing value has two very different 
explanations depending on where the trip started:

- **Non-airport pickup + missing Airport_fee** → makes sense as $0, since the fee 
  was never supposed to apply to this trip in the first place. Safe to fill with 0.

- **JFK/LaGuardia pickup + missing Airport_fee** → this is a real gap. The fee 
  should have been charged, but the value wasn't recorded. Filling this with 0 
  would incorrectly suggest no fee applied, when in reality we simply don't know 
  the amount. These are left as NULL instead of guessing.

In [177]:
df_clean.loc[df_clean['Airport_fee'].isna() & (df_clean['PULocationID'].isin([132, 138]) == False), 'Airport_fee'] = 0

In [178]:
# Add new column as pickup_hour.
df_clean['pickup_hour'] = df_clean['tpep_pickup_datetime'].dt.hour

In [179]:
df_clean.isnull().sum()

VendorID                      0
tpep_pickup_datetime          0
tpep_dropoff_datetime         0
passenger_count          964283
trip_distance            109359
RatecodeID                    0
store_and_fwd_flag            0
PULocationID                  0
DOLocationID                  0
payment_type                  0
fare_amount               31829
extra                         0
mta_tax                       0
tip_amount                    0
tolls_amount                  0
improvement_surcharge         0
total_amount                  0
congestion_surcharge     952045
Airport_fee                1032
cbd_congestion_fee            0
is_cancelled                  0
is_dispute_adjustment         0
fare_per_mile            109359
is_fare_outlier               0
is_invalid_location           0
pickup_hour                   0
dtype: int64

In [180]:
df_clean.shape

(4050863, 26)

In [181]:
Raw_Data_Shape = df.shape[0]
Cleaned_Data_Shape = df_clean.shape[0]
Removed_Data = Raw_Data_Shape - Cleaned_Data_Shape

pct_removed = (Removed_Data / Raw_Data_Shape) * 100

print("Original rows:", Raw_Data_Shape)
print("Final rows:", Cleaned_Data_Shape)
print("Rows removed:", Removed_Data)
print('Percentage removed:', round(pct_removed, 2),'%')

Original rows: 4090836
Final rows: 4050863
Rows removed: 39973
Percentage removed: 0.98 %


**Data Importing of Table 2 (Taxi Zone Lookup)**

**Loading CSV Data to PostgreSQL :-**

In [182]:
# file_path = r"C:\Users\hp\OneDrive\Documents\DA Course\End to End Projects\Project 6\taxi_zone_lookup.csv"

# table_name = "taxi_zone_lookup"

# df = pd.read_csv(file_path,low_memory=False)
# df.to_sql(table_name, engine, if_exists="replace", index=False)

# print(f"Loaded {table_name}: {len(df)} rows")

**Importing Data in python from PostgreSQL :-**

In [183]:
conn = psycopg2.connect(
    dbname="Taxi_Analysis_db",
    user="postgres",
    password="Canada@040201",
    host="localhost",
    port="5432"
)

sql_query="""
          SELECT table_name AS name FROM information_schema.tables
          WHERE table_schema = 'public'
"""


In [184]:
table_name = "taxi_zone_lookup"
data = pd.read_sql(f"SELECT * FROM {table_name}", conn) 

**Data Overview in Table 2 (Taxi Zone Lookup :-**

In [185]:
data.head()

,LocationID,Borough,Zone,service_zone
0,1,EWR,Newark Airport,EWR
1,2,Queens,Jamaica Bay,Boro Zone
2,3,Bronx,Allerton/Pelham Gardens,Boro Zone
3,4,Manhattan,Alphabet City,Yellow Zone
4,5,Staten Island,Arden Heights,Boro Zone


In [186]:
data.tail()

,LocationID,Borough,Zone,service_zone
260,261,Manhattan,World Trade Center,Yellow Zone
261,262,Manhattan,Yorkville East,Yellow Zone
262,263,Manhattan,Yorkville West,Yellow Zone
263,264,Unknown,NaN,NaN
264,265,NaN,Outside of NYC,NaN


In [187]:
data.shape

(265, 4)

In [188]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 265 entries, 0 to 264
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   LocationID    265 non-null    int64
 1   Borough       264 non-null    str  
 2   Zone          264 non-null    str  
 3   service_zone  263 non-null    str  
dtypes: int64(1), str(3)
memory usage: 8.4 KB


In [189]:
data.isnull().sum()

LocationID      0
Borough         1
Zone            1
service_zone    2
dtype: int64

In [190]:
data.duplicated().sum()

np.int64(0)

**Data Cleaning in Table 2 (Taxi Zone Lookup) :-**

In [191]:
data.dropna(inplace=True)
data.shape

(263, 4)

**Feature Engineering + Data Analysis + KPI's**

In [192]:
# Check if dropoff time is before/equal to pickup time (impossible trip)
df_clean['trip_duration_min'] = (
    df_clean['tpep_dropoff_datetime'] - df_clean['tpep_pickup_datetime']
).dt.total_seconds()/60

# Flag rows where dropoff <= pickup (0 or negative duration)
df_clean['is_invalid_time'] = df_clean['trip_duration_min'] <= 0

print("Invalid time rows:", df_clean['is_invalid_time'].sum())

Invalid time rows: 51616


In [193]:
# Look at a few before deciding what to do with them
df_clean[df_clean['is_invalid_time']][
    ['tpep_pickup_datetime', 'tpep_dropoff_datetime', 'trip_duration_min', 'trip_distance', 'fare_amount']
].tail(10)

,tpep_pickup_datetime,tpep_dropoff_datetime,trip_duration_min,trip_distance,fare_amount
3665495,2026-05-18 08:45:22,2026-05-18 08:45:22,0.00,2.70,30.78
3744527,2026-05-20 20:54:00,2026-05-20 20:54:00,0.00,NaN,13.37
3756976,2026-05-21 06:25:00,2026-05-21 06:25:00,0.00,NaN,76.07
3808250,2026-05-22 22:42:00,2026-05-22 22:42:00,0.00,NaN,22.52
3874405,2026-05-25 00:24:09,2026-05-25 00:24:09,0.00,NaN,15.89
3954610,2026-05-28 10:22:00,2026-05-28 10:22:00,0.00,0.01,NaN
4006143,2026-05-29 22:36:37,2026-05-29 22:36:37,0.00,NaN,23.19
4011337,2026-05-30 00:24:39,2026-05-30 00:24:39,0.00,NaN,13.51
4084115,2026-05-31 18:12:00,2026-05-31 18:12:00,0.00,NaN,5.90
4084536,2026-05-31 18:49:00,2026-05-31 18:49:00,0.00,NaN,9.32


In [194]:
Suspicious_duration = df_clean[df_clean['is_invalid_time']].copy()
print("Total Suspicious trip duration:", Suspicious_duration.shape[0])

Total Suspicious trip duration: 51616


In [195]:
df_clean.loc[df_clean['is_invalid_time'], 'trip_duration_min'] = pd.NA

In [196]:
df_clean[(df_clean[['trip_duration_min', 'trip_distance', 'fare_amount']]==0).all(axis=1)]

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,is_cancelled,is_dispute_adjustment,fare_per_mile,is_fare_outlier,is_invalid_location,pickup_hour,trip_duration_min,is_invalid_time


In [197]:
df_clean['is_invalid_date'] = df_clean['tpep_pickup_datetime'] < '2026-01-01'
print(df_clean['is_invalid_date'].sum())

1


In [198]:
df_clean = df_clean[~df_clean['is_invalid_date']].copy()

In [199]:
# 8) Check if total_amount matches the sum of its components
component_cols = ['fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount',
                   'improvement_surcharge', 'congestion_surcharge', 'Airport_fee','cbd_congestion_fee']

df_clean['expected_total'] = df_clean[component_cols].sum(axis=1, skipna=False)
df_clean['total_diff'] = (df_clean['total_amount'] - df_clean['expected_total']).abs()

# Allow a small tolerance (rounding) rather than exact match
df_clean['is_total_mismatch'] = df_clean['total_diff'] > 0.05

print("Mismatched total_amount rows:", df_clean['is_total_mismatch'].sum())

Mismatched total_amount rows: 545117


In [200]:
# Look at a sample before deciding what to do
df_clean[df_clean['is_total_mismatch']][
    ['fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount',
     'improvement_surcharge', 'congestion_surcharge', 'Airport_fee',
     'total_amount', 'expected_total', 'total_diff']
].head(10)

,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,congestion_surcharge,Airport_fee,total_amount,expected_total,total_diff
2,19.10,4.25,0.50,3.73,0.00,1.00,2.50,0.00,28.58,31.83,3.25
3,9.30,4.25,0.50,3.00,0.00,1.00,2.50,0.00,18.05,21.30,3.25
4,7.20,0.00,0.50,2.44,0.00,1.00,2.50,0.00,14.64,13.64,1.00
6,9.30,3.50,0.50,2.85,0.00,1.00,2.50,0.00,17.15,19.65,2.50
13,5.80,4.25,0.50,0.00,0.00,1.00,2.50,0.00,11.55,14.80,3.25
16,19.80,4.25,0.50,5.10,0.00,1.00,2.50,0.00,30.65,33.90,3.25
17,9.30,4.25,0.50,3.75,0.00,1.00,2.50,0.00,18.80,22.05,3.25
20,70.00,4.50,0.50,12.51,7.46,1.00,2.50,2.00,95.97,100.47,4.50
43,8.60,4.25,0.50,2.85,0.00,1.00,2.50,0.00,17.20,20.45,3.25
44,10.00,4.25,0.50,3.15,0.00,1.00,2.50,0.00,18.90,22.15,3.25


**total_amount (mismatched rows)** --> Leave as missing (NA), don't guess.

After checking sum of fare amount with rest amounts, total_amount must be same. If not, we will replace total amount with NaN (Missing Values). [To fix the errors].

Why we didn't replace total amount with expected total= If any component value (fare amount with rest amounts) was recorded wrong then expected total will also calculate incorrectly.

In [201]:
df_clean.loc[df_clean['is_total_mismatch'], 'total_amount'] = pd.NA

In [202]:
df_clean.isnull().sum()

VendorID                      0
tpep_pickup_datetime          0
tpep_dropoff_datetime         0
passenger_count          964283
trip_distance            109359
RatecodeID                    0
store_and_fwd_flag            0
PULocationID                  0
DOLocationID                  0
payment_type                  0
fare_amount               31829
extra                         0
mta_tax                       0
tip_amount                    0
tolls_amount                  0
improvement_surcharge         0
total_amount             545117
congestion_surcharge     952045
Airport_fee                1032
cbd_congestion_fee            0
is_cancelled                  0
is_dispute_adjustment         0
fare_per_mile            109359
is_fare_outlier               0
is_invalid_location           0
pickup_hour                   0
trip_duration_min         51616
is_invalid_time               0
is_invalid_date               0
expected_total           960981
total_diff               960981
is_total

In [203]:
# Check for nonzero tip_amount on cash payments (payment_type = 2)
df_clean['is_cash_tip_error'] = (df_clean['payment_type'] == 2) & (df_clean['tip_amount'] > 0)

print("Cash payments with a recorded tip:", df_clean['is_cash_tip_error'].sum())

Cash payments with a recorded tip: 25


In [204]:
# Look at a sample before deciding
df_clean[df_clean['is_cash_tip_error']][
    ['payment_type', 'tip_amount']
]

,payment_type,tip_amount
174074,2,5.68
348014,2,9.81
362248,2,8.07
426459,2,20.20
499285,2,2.31
619635,2,6.59
641456,2,4.91
770468,2,2.81
978031,2,4.91
1089066,2,6.40


In [205]:
df_clean.loc[df_clean['is_cash_tip_error'], 'tip_amount'] = pd.NA

In [206]:
# Statistical summary to catch anything the logic-based checks missed
df_clean.describe().round(2)

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,fare_per_mile,pickup_hour,trip_duration_min,expected_total,total_diff
count,"4,050,862.00",4050862,4050862,"3,086,579.00","3,941,503.00","4,050,862.00","4,050,862.00","4,050,862.00","4,050,862.00","4,019,033.00","4,050,862.00","4,050,862.00","4,050,837.00","4,050,862.00","4,050,862.00","3,505,745.00","3,098,817.00","4,049,830.00","4,050,862.00","3,941,503.00","4,050,862.00","3,999,246.00","3,089,881.00","3,089,881.00"
mean,1.87,2026-05-16 10:36:35.706343,2026-05-16 10:55:17.423244,1.25,5.05,27.50,162.02,161.30,0.87,21.31,1.13,0.49,2.95,0.51,0.96,30.78,2.23,0.12,0.50,23.44,14.15,18.94,30.18,0.54
min,1.00,2026-04-30 23:47:18,2026-04-30 23:53:05,1.00,0.01,1.00,1.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.02,0.00,0.00
25%,2.00,2026-05-08 21:15:26.250000,2026-05-08 21:32:25.250000,1.00,1.10,1.00,114.00,107.00,1.00,10.00,0.00,0.50,0.00,0.00,1.00,17.94,2.50,0.00,0.00,5.82,10.00,8.80,17.45,0.00
50%,2.00,2026-05-16 01:40:14.500000,2026-05-16 01:55:42,1.00,1.92,1.00,161.00,162.00,1.00,16.30,0.00,0.50,2.25,0.00,1.00,24.40,2.50,0.00,0.75,7.66,15.00,14.62,23.00,0.00
75%,2.00,2026-05-23 15:36:47,2026-05-23 15:54:21,1.00,3.89,99.00,234.00,234.00,1.00,26.45,2.50,0.50,4.08,0.00,1.00,35.56,2.50,0.00,0.75,10.09,19.00,23.33,33.05,0.00
max,7.00,2026-06-01 00:20:35,2026-06-02 12:53:42,9.00,"307,491.47",99.00,263.00,263.00,4.00,"5,525.99",15.25,4.75,222.00,145.60,2.50,"5,530.74",2.50,27.00,0.75,"65,500.00",23.00,"9,960.15","5,530.74",5.25
std,0.73,NaN,NaN,0.63,492.69,43.46,67.00,70.49,0.59,17.47,1.74,0.05,3.89,2.04,0.19,21.51,0.77,0.51,0.35,253.37,6.00,26.65,22.55,1.19


In [207]:
# Convert trip_distance from miles to kilometers
df_clean['trip_distance_km'] = df_clean['trip_distance'] * 1.60934

In [208]:
# Check trip_distance_km directly for extreme values (independent of fare)
# NYC to the farthest reasonable point (e.g. Montauk, or a cross-country outlier check) is well under 200 miles
distance_threshold = df_clean['trip_distance_km'].quantile(0.999983)
print("99.9th percentile trip_distance_km:", distance_threshold)

df_clean['is_distance_outlier'] = df_clean['trip_distance_km'] > distance_threshold
print("Distance outlier rows:", df_clean['is_distance_outlier'].sum())

99.9th percentile trip_distance_km: 342.02723303054853
Distance outlier rows: 68


In [209]:
df_clean.loc[df_clean['is_distance_outlier'], 'trip_distance_km'] = pd.NA

In [210]:
df_clean.isnull().sum()

VendorID                      0
tpep_pickup_datetime          0
tpep_dropoff_datetime         0
passenger_count          964283
trip_distance            109359
RatecodeID                    0
store_and_fwd_flag            0
PULocationID                  0
DOLocationID                  0
payment_type                  0
fare_amount               31829
extra                         0
mta_tax                       0
tip_amount                   25
tolls_amount                  0
improvement_surcharge         0
total_amount             545117
congestion_surcharge     952045
Airport_fee                1032
cbd_congestion_fee            0
is_cancelled                  0
is_dispute_adjustment         0
fare_per_mile            109359
is_fare_outlier               0
is_invalid_location           0
pickup_hour                   0
trip_duration_min         51616
is_invalid_time               0
is_invalid_date               0
expected_total           960981
total_diff               960981
is_total

In [211]:
df_clean.info()

<class 'pandas.DataFrame'>
Index: 4050862 entries, 0 to 4090835
Data columns (total 35 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   VendorID               int64         
 1   tpep_pickup_datetime   datetime64[us]
 2   tpep_dropoff_datetime  datetime64[us]
 3   passenger_count        Int64         
 4   trip_distance          float64       
 5   RatecodeID             Int64         
 6   store_and_fwd_flag     str           
 7   PULocationID           int64         
 8   DOLocationID           int64         
 9   payment_type           int64         
 10  fare_amount            float64       
 11  extra                  float64       
 12  mta_tax                float64       
 13  tip_amount             float64       
 14  tolls_amount           float64       
 15  improvement_surcharge  float64       
 16  total_amount           float64       
 17  congestion_surcharge   float64       
 18  Airport_fee            float64       


In [212]:
pd.set_option('display.float_format','{:,.2f}'.format)
df_clean.describe()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,fare_per_mile,pickup_hour,trip_duration_min,expected_total,total_diff,trip_distance_km
count,"4,050,862.00",4050862,4050862,"3,086,579.00","3,941,503.00","4,050,862.00","4,050,862.00","4,050,862.00","4,050,862.00","4,019,033.00","4,050,862.00","4,050,862.00","4,050,837.00","4,050,862.00","4,050,862.00","3,505,745.00","3,098,817.00","4,049,830.00","4,050,862.00","3,941,503.00","4,050,862.00","3,999,246.00","3,089,881.00","3,089,881.00","3,941,435.00"
mean,1.87,2026-05-16 10:36:35.706343,2026-05-16 10:55:17.423244,1.25,5.05,27.50,162.02,161.30,0.87,21.31,1.13,0.49,2.95,0.51,0.96,30.78,2.23,0.12,0.50,23.44,14.15,18.94,30.18,0.54,5.55
min,1.00,2026-04-30 23:47:18,2026-04-30 23:53:05,1.00,0.01,1.00,1.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.02,0.00,0.00,0.02
25%,2.00,2026-05-08 21:15:26.250000,2026-05-08 21:32:25.250000,1.00,1.10,1.00,114.00,107.00,1.00,10.00,0.00,0.50,0.00,0.00,1.00,17.94,2.50,0.00,0.00,5.82,10.00,8.80,17.45,0.00,1.77
50%,2.00,2026-05-16 01:40:14.500000,2026-05-16 01:55:42,1.00,1.92,1.00,161.00,162.00,1.00,16.30,0.00,0.50,2.25,0.00,1.00,24.40,2.50,0.00,0.75,7.66,15.00,14.62,23.00,0.00,3.09
75%,2.00,2026-05-23 15:36:47,2026-05-23 15:54:21,1.00,3.89,99.00,234.00,234.00,1.00,26.45,2.50,0.50,4.08,0.00,1.00,35.56,2.50,0.00,0.75,10.09,19.00,23.33,33.05,0.00,6.26
max,7.00,2026-06-01 00:20:35,2026-06-02 12:53:42,9.00,"307,491.47",99.00,263.00,263.00,4.00,"5,525.99",15.25,4.75,222.00,145.60,2.50,"5,530.74",2.50,27.00,0.75,"65,500.00",23.00,"9,960.15","5,530.74",5.25,320.74
std,0.73,NaN,NaN,0.63,492.69,43.46,67.00,70.49,0.59,17.47,1.74,0.05,3.89,2.04,0.19,21.51,0.77,0.51,0.35,253.37,6.00,26.65,22.55,1.19,6.56


In [213]:
# 1 - Total number of taxi trips 
Total_Trips = len(df_clean)
print('Total number of Taxi Trips :',Total_Trips)

Total number of Taxi Trips : 4050862


In [214]:
# 2 -  Total Revenue
Total_Revenue = df_clean['total_amount'].fillna(df_clean['expected_total']).sum().round(2)
print('Total Revenue :',Total_Revenue)

Total Revenue : 124356507.26


In [215]:
# 3 -  Average trip Amount
Average_Fare_Amount = df_clean['total_amount'].fillna(df_clean['expected_total']).mean().round(2)
print('Average Fare Amount Per Trip :',Average_Fare_Amount)

Average Fare Amount Per Trip : 30.7


In [216]:
# 4 -  Average tip per trip
Average_Tip_Amount = df_clean['tip_amount'].mean().round(2)
print('Average Tip Amount :',Average_Tip_Amount)

Average Tip Amount : 2.95


In [217]:
# 5 -  Average trip distance in miles.
Average_Trip_Distance = df_clean['trip_distance'].mean().round(2)
print('Average Trip Distance :',Average_Trip_Distance)

Average Trip Distance : 5.05


In [218]:
# 6 -  Average minutes between pickup and dropoff.
Average_Trip_Duration = df_clean['trip_duration_min'].mean().round(2)
print('Average Trip Duration In Min :',Average_Trip_Duration)

Average Trip Duration In Min : 18.94


In [219]:
# 7 -  Revenue per KM
Revenue_Per_KM = (Total_Revenue/df_clean['trip_distance_km'].sum()).round(2)
print('Revenue Per KM :',Revenue_Per_KM)

Revenue Per KM : 5.68


In [220]:
# 8 -  Tip amount as a share of trip value.
Tip_Rate = ((df_clean['tip_amount'].sum()/Total_Revenue)*100).round(2)
print('Tip Rate :',Tip_Rate, '%')

Tip Rate : 9.62 %


In [221]:
# 9 -  Share of trips associated with airport rate codes.
Airport_Trip_Share_Pct = (((df_clean['Airport_fee']>0).sum()/len(df_clean))*100).round(2)
print('Airport Trip Share(%) :',Airport_Trip_Share_Pct)

Airport Trip Share(%) : 6.08


In [222]:
# 10 -  Most Used Payment Method.
Payment_Count = df_clean['payment_type'].value_counts()
Most_Used_Payment_Method = Payment_Count.idxmax()
print('Most Used Payment Mathod :',Most_Used_Payment_Method)

Most Used Payment Mathod : 1


Here 1 = Credit card, 2 = Cash, 3 = No charge, 4 = Dispute

In [223]:
# 11 -  Busiest Day of Week
Day_Count = df_clean['tpep_pickup_datetime'].dt.day_name().value_counts()
Busiest_Day = Day_Count.idxmax()
print('Busiest Day of Week :',Busiest_Day)

Busiest Day of Week : Friday


In [224]:
# 12 -  Vendor Share.
Vendor_Count = df_clean['VendorID'].value_counts()
Vendor_Share = ((Vendor_Count/len(df_clean))*100).round(2)
Top_Vendor_Share = Vendor_Share.idxmax()
print('Top Vendor Id :',Vendor_Share.idxmax())
print('Top Vendor Share :',Vendor_Share.max(),'%')

Top Vendor Id : 2
Top Vendor Share : 78.83 %


1 = Creative Mobile Technologies, LLC

2 = Curb Mobility, LLC

6 = Myle Technologies Inc

7 = Helix

In [225]:
# 13 -  Peak hour of day with Trip Counts.
Hourly_Trips = df_clean['pickup_hour'].value_counts()
Peak_Hour = Hourly_Trips.idxmax()
Total_Trips_In_Peak_Hour = Hourly_Trips.max()
print('Peak Hour :',Peak_Hour)
print('Total Trips Count in Peak Hour :',Total_Trips_In_Peak_Hour)


Peak Hour : 18
Total Trips Count in Peak Hour : 281166


**Create dimension tables by using Data Dictionary :-**

**Vendor Dimension Table**

In [226]:
Vendor = {'VendorID':[1,2,6,7],
          'Vendor_Names':['Creative Mobile Technologies, LLC','Curb Mobility, LLC','Myle Technologies Inc','Helix']}

Vendor_Lookup = pd.DataFrame(Vendor)
Vendor_Lookup

,VendorID,Vendor_Names
0,1,"Creative Mobile Technologies, LLC"
1,2,"Curb Mobility, LLC"
2,6,Myle Technologies Inc
3,7,Helix


In [227]:
Vendor_Lookup.to_csv('dim_vendor.csv',index=False)

**Rate Code Dimension Table**

In [228]:
Rate_Code = {'RatecodeID':[1,2,3,4,5,6,99],
             'Rate_Code':['Standard rate','JFK','Newark','Nassau or Westchester','Negotiated fare','Group ride','Null/unknown']}
Rate_Code_Lookup = pd.DataFrame(Rate_Code)
Rate_Code_Lookup

,RatecodeID,Rate_Code
0,1,Standard rate
1,2,JFK
2,3,Newark
3,4,Nassau or Westchester
4,5,Negotiated fare
5,6,Group ride
6,99,Null/unknown


In [229]:
Rate_Code_Lookup.to_csv('dim_rate_code.csv',index=False)

**Store & FWD Flag Dimension Table**

In [230]:
Sff ={'store_and_fwd_flag':['Y','N'],
      'Flags':['store and forward trip','not a store and forward trip']}
Store_and_FWD_Flag_Lookup = pd.DataFrame(Sff)
Store_and_FWD_Flag_Lookup

,store_and_fwd_flag,Flags
0,Y,store and forward trip
1,N,not a store and forward trip


In [231]:
Store_and_FWD_Flag_Lookup.to_csv('dim_store_and_fWD_flag.csv',index=False)

**Payment Type Dimension Table**

In [232]:
df_clean.columns

Index(['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime',
       'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag',
       'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra',
       'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge',
       'total_amount', 'congestion_surcharge', 'Airport_fee',
       'cbd_congestion_fee', 'is_cancelled', 'is_dispute_adjustment',
       'fare_per_mile', 'is_fare_outlier', 'is_invalid_location',
       'pickup_hour', 'trip_duration_min', 'is_invalid_time',
       'is_invalid_date', 'expected_total', 'total_diff', 'is_total_mismatch',
       'is_cash_tip_error', 'trip_distance_km', 'is_distance_outlier'],
      dtype='str')

In [233]:
Payment_Type = {'payment_type':[0,1,2,3,4,5,6],
                'Payment_Method':['Flex Fare trip','Credit card','Cash','No charge','Dispute','Unknown','Voided trip']}
Payment_Type_Lookup = pd.DataFrame(Payment_Type)
Payment_Type_Lookup

,payment_type,Payment_Method
0,0,Flex Fare trip
1,1,Credit card
2,2,Cash
3,3,No charge
4,4,Dispute
5,5,Unknown
6,6,Voided trip


In [234]:
Payment_Type_Lookup.to_csv('dim_payment_type.csv',index=False)

**Taxi Zone Dimension Table**

In [235]:
data.to_csv('dim_zone.csv',index=False)

**Yellow Trip Fact Table**

In [236]:
df_clean.to_csv('fact_yellow_trip.csv',index=False)

**Exception Table**

In [237]:
# 1. Names of the 3 outlier flags
flag_cols = ['is_cash_tip_error', 'is_distance_outlier', 'is_fare_outlier']

# 2. Negative rows from the original df
exception_mask = (df[negative_cols] < 0).any(axis=1)
df_exceptions = df.loc[exception_mask].copy()

# 3. Outlier rows from df_clean, added under the negative rows
df_out = df_clean[df_clean[flag_cols].fillna(False).any(axis=1)]
df_exceptions = pd.concat([df_exceptions, df_out], ignore_index=True)

# 4. Empty flags become False
df_exceptions[flag_cols] = df_exceptions[flag_cols].fillna(False)

# 5. Check the counts
print('Exception rows:', len(df_exceptions))
print(df_exceptions['is_cancelled'].value_counts())
print(df_exceptions['is_dispute_adjustment'].value_counts())
print(df_exceptions[flag_cols].sum())

# 6. Save to PostgreSQL
df_exceptions.to_sql('exception_trips', engine, if_exists='replace', index=False)

Exception rows: 46874
is_cancelled
False    45102
True      1772
Name: count, dtype: int64
is_dispute_adjustment
False    37830
True      9044
Name: count, dtype: int64
is_cash_tip_error         25
is_distance_outlier       68
is_fare_outlier        31829
dtype: object


174

In [238]:
# Save to PostgreSQL
df_exceptions.to_csv('df_exceptions.csv',index=False)

In [239]:
df_exceptions.shape 

(46874, 35)

In [240]:
df_exceptions.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,is_cancelled,is_dispute_adjustment,fare_per_mile,is_fare_outlier,is_invalid_location,pickup_hour,trip_duration_min,is_invalid_time,is_invalid_date,expected_total,total_diff,is_total_mismatch,is_cash_tip_error,trip_distance_km,is_distance_outlier
0,2,2026-05-01 00:13:36,2026-05-01 00:25:27,1,2.06,1,N,230,141,4,-12.80,-1.00,-0.50,0.00,0.00,-1.00,-18.55,-2.50,0.00,-0.75,False,True,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,False
1,2,2026-05-01 00:21:41,2026-05-01 00:22:46,1,0.11,1,N,233,233,4,-3.70,-1.00,-0.50,0.00,0.00,-1.00,-9.45,-2.50,0.00,-0.75,False,True,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,False
2,2,2026-05-01 00:37:49,2026-05-01 00:59:10,2,6.16,1,N,87,100,4,-30.30,-1.00,-0.50,0.00,0.00,-1.00,-36.05,-2.50,0.00,-0.75,False,True,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,False
3,2,2026-05-01 00:02:26,2026-05-01 00:12:49,1,1.88,1,N,170,113,4,-12.10,-1.00,-0.50,0.00,0.00,-1.00,-17.85,-2.50,0.00,-0.75,False,True,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,False
4,2,2026-05-01 00:35:03,2026-05-01 00:55:00,1,5.14,1,N,100,74,4,-24.00,-1.00,-0.50,0.00,0.00,-1.00,-29.75,-2.50,0.00,-0.75,False,True,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,False


In [241]:
df_exceptions['is_cancelled'].value_counts()

is_cancelled
False    45102
True      1772
Name: count, dtype: int64

In [242]:
print(df_exceptions['is_dispute_adjustment'].sum())

9044
